In [ ]:
import os
import json
import librosa
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from datasets import Dataset, Audio

# Configuration
MAIN_PATH = "/kaggle/input/datasets/iveeaten3223times/multilingual-indian-speech-data"
EXTRA_PATH = "/kaggle/input/datasets/adhithyasash1/indic-deepfake-challenge-extra-dataset/extra-dataset"
SAMPLE_AUDIO_COUNT = 200  # Number of audio files to sample per category for physical feature extraction
SEED = 42

def load_local_main_dataset():
    metadata_dir = os.path.join(MAIN_PATH, "metadata")
    audio_dir = os.path.join(MAIN_PATH, "audio")
    
    # Load train and test metadata CSVs directly
    train_df = pd.read_csv(os.path.join(metadata_dir, "train.csv"))
    test_df = pd.read_csv(os.path.join(metadata_dir, "test.csv"))
    
    # Map the nested CSV paths to the actual flat local file paths
    train_df["audio"] = train_df["audio_path"].apply(lambda p: os.path.join(audio_dir, os.path.basename(p)))
    test_df["audio"] = test_df["audio_path"].apply(lambda p: os.path.join(audio_dir, os.path.basename(p)))
    
    # Convert to Hugging Face Dataset objects
    train_ds = Dataset.from_pandas(train_df)
    test_ds = Dataset.from_pandas(test_df)
    
    # Cast the audio path strings as Audio features
    train_ds = train_ds.cast_column("audio", Audio(sampling_rate=16000))
    test_ds = test_ds.cast_column("audio", Audio(sampling_rate=16000))
    
    return train_ds, test_ds

def run_eda():
    print("="*60)
    print(" 🚀 STARTING COMPREHENSIVE DATASET ANALYSIS")
    print("="*60)
    
    # -------------------------------------------------------------
    # 1. Load Datasets
    # -------------------------------------------------------------
    print("\n[1/5] Loading datasets metadata...")
    main_train, main_test = None, None
    extra_ds = None
    main_df, extra_df = pd.DataFrame(), pd.DataFrame()
    
    try:
        main_train, main_test = load_local_main_dataset()
        print(f"✅ Main dataset loaded successfully using flat CSV mapping!")
        print(f"   Train samples: {len(main_train)} | Test samples: {len(main_test)}")
        print(f"   Columns in Main train: {main_train.column_names}")
        main_df = pd.DataFrame(main_train.remove_columns("audio"))
    except Exception as e:
        print(f"❌ Error loading main dataset: {e}")
        
    try:
        extra_ds = load_dataset("audiofolder", data_dir=EXTRA_PATH)
        print(f"✅ Extra dataset loaded successfully from: {EXTRA_PATH}")
        print(f"   Columns in 'train': {extra_ds['train'].column_names}")
        extra_df = pd.DataFrame(extra_ds["train"].remove_columns("audio"))
    except Exception as e:
        print(f"❌ Error loading extra dataset: {e}")

    # -------------------------------------------------------------
    # 2. General Class and Language Distributions
    # -------------------------------------------------------------
    print("\n" + "="*50)
    print(" [2/5] GENERAL STATISTICS")
    print("="*50)
    
    if not main_df.empty:
        print("\n--- Main Dataset Label Distribution ---")
        if "is_tts" in main_df.columns:
            counts = main_df["is_tts"].value_counts()
            pcts = main_df["is_tts"].value_counts(normalize=True) * 100
            for val, cnt in counts.items():
                lbl = "Synthetic (1)" if int(val) == 1 else "Real Human (0)"
                print(f"   - {lbl}: {cnt} ({pcts[val]:.2f}%)")
        else:
            print("   ⚠️ Column 'is_tts' not found in main dataset metadata.")

    if not extra_df.empty:
        print("\n--- Extra Dataset Label Distribution ---")
        if "is_tts" in extra_df.columns:
            counts = extra_df["is_tts"].value_counts()
            pcts = extra_df["is_tts"].value_counts(normalize=True) * 100
            for val, cnt in counts.items():
                lbl = "Synthetic (1)" if int(val) == 1 else "Real Human (0)"
                print(f"   - {lbl}: {cnt} ({pcts[val]:.2f}%)")

    # Language Breakdown Comparison
    print("\n--- Language Distribution Comparison ---")
    lang_comp = {}
    
    if not main_df.empty and "language" in main_df.columns:
        for lang, group in main_df.groupby("language"):
            real = (group["is_tts"] == 0).sum() if "is_tts" in group.columns else 0
            fake = (group["is_tts"] == 1).sum() if "is_tts" in group.columns else 0
            lang_comp[lang] = {"Main_Real": real, "Main_Fake": fake, "Extra_Real": 0, "Extra_Fake": 0}
            
    if not extra_df.empty and "language" in extra_df.columns:
        for lang, group in extra_df.groupby("language"):
            real = (group["is_tts"] == 0).sum() if "is_tts" in group.columns else 0
            fake = (group["is_tts"] == 1).sum() if "is_tts" in group.columns else 0
            if lang not in lang_comp:
                lang_comp[lang] = {"Main_Real": 0, "Main_Fake": 0, "Extra_Real": 0, "Extra_Fake": 0}
            lang_comp[lang]["Extra_Real"] = real
            lang_comp[lang]["Extra_Fake"] = fake

    if lang_comp:
        comp_df = pd.DataFrame.from_dict(lang_comp, orient="index")
        comp_df["Total"] = comp_df.sum(axis=1)
        print(comp_df.sort_values(by="Total", ascending=False).to_markdown())
    else:
        print("   ⚠️ No language statistics could be aggregated.")

    # -------------------------------------------------------------
    # 3. Text Transcription Statistics
    # -------------------------------------------------------------
    print("\n" + "="*50)
    print(" [3/5] TRANSCRIPTION METADATA STATISTICS")
    print("="*50)
    
    for name, df in [("Main", main_df), ("Extra", extra_df)]:
        if df.empty or "text" not in df.columns or "is_tts" not in df.columns:
            continue
        print(f"\n--- {name} Dataset Text Metrics (Real vs. Synthetic) ---")
        df["char_len"] = df["text"].apply(lambda x: len(str(x)))
        df["word_cnt"] = df["text"].apply(lambda x: len(str(x).split()))
        df["uniq_char"] = df["text"].apply(lambda x: len(set(str(x))) / (len(str(x)) + 1))
        
        metrics = df.groupby("is_tts")[["char_len", "word_cnt", "uniq_char"]].agg(["mean", "std", "median"])
        print(metrics.to_markdown())

    # -------------------------------------------------------------
    # 4. Audio Quality & Acoustic Properties (Sampled Analysis)
    # -------------------------------------------------------------
    print("\n" + "="*50)
    print(" [4/5] ACOUSTIC / AUDIO ANALYSIS (SAMPLED SUBSET)")
    print("="*50)
    
    acoustic_results = []
    
    def process_sampled_ds(ds_name, split_dataset):
        if "is_tts" not in split_dataset.column_names:
            print(f"⚠️ Skipping audio analysis for {ds_name}: is_tts column not found.")
            return
            
        labels = np.array(split_dataset["is_tts"])
        for c in [0, 1]:
            c_indices = np.where(labels == c)[0]
            if len(c_indices) == 0:
                continue
            np.random.seed(SEED)
            sampled_idx = np.random.choice(c_indices, size=min(SAMPLE_AUDIO_COUNT, len(c_indices)), replace=False)
            
            lbl_str = "Synthetic" if c == 1 else "Real"
            desc = f"Analyzing {ds_name} {lbl_str} audio"
            
            for idx in tqdm(sampled_idx, desc=desc):
                item = split_dataset[int(idx)]
                audio = item["audio"]
                
                # Load raw data
                arr = np.asarray(audio["array"], dtype=np.float32)
                sr = int(audio["sampling_rate"])
                
                # Duration & stats
                duration = len(arr) / sr
                peak = float(np.max(np.abs(arr)))
                rms = float(np.sqrt(np.mean(arr**2)))
                
                # Pitch
                try:
                    f0 = librosa.yin(arr, fmin=librosa.note_to_hz("C2"), fmax=librosa.note_to_hz("C7"), sr=sr)
                    f0 = f0[~np.isnan(f0)]
                    f0_mean = float(f0.mean()) if f0.size > 0 else 0.0
                    f0_std  = float(f0.std()) if f0.size > 0 else 0.0
                except Exception:
                    f0_mean, f0_std = 0.0, 0.0
                    
                # Spectral metrics
                spec_flat = float(np.mean(librosa.feature.spectral_flatness(y=arr)))
                spec_cent = float(np.mean(librosa.feature.spectral_centroid(y=arr, sr=sr)))
                
                acoustic_results.append({
                    "Dataset": ds_name,
                    "Label": lbl_str,
                    "Duration": duration,
                    "SR": sr,
                    "Peak": peak,
                    "RMS": rms,
                    "F0_Mean": f0_mean,
                    "F0_Std": f0_std,
                    "Flatness": spec_flat,
                    "Centroid": spec_cent
                })

    if main_train:
        process_sampled_ds("Main", main_train)
    if extra_ds and "train" in extra_ds:
        process_sampled_ds("Extra", extra_ds["train"])
        
    if acoustic_results:
        ac_df = pd.DataFrame(acoustic_results)
        print("\n--- Mean Acoustic Metrics by Dataset and Label ---")
        summary = ac_df.groupby(["Dataset", "Label"]).mean()
        print(summary.to_markdown())
        
        print("\n--- Audio Sample Rate Counts ---")
        print(ac_df.groupby(["Dataset"])["SR"].value_counts().to_markdown())
    else:
        print("⚠️ No audio files could be processed.")

    # -------------------------------------------------------------
    # 5. Conclusions & Next Steps
    # -------------------------------------------------------------
    print("\n" + "="*50)
    print(" [5/5] EDA COMPLETED")
    print("="*50)

run_eda()